# 98 · Pipeline passo a passo (uma célula por etapa)

Este notebook faz o mesmo que o `99_orquestrador`, mas **cada etapa roda numa célula própria**, com os resultados dela logo abaixo. Serve para acompanhar o pipeline etapa por etapa, olhar os resultados antes de seguir e rodar de novo só a etapa que falhou.

| célula | etapa | variável criada |
|---|---|---|
| Etapa 1 | entrada, validação e divisão treino/teste | `entrada` |
| Etapas 2 e 3 | pré-filtros e Boruta | `selecao` |
| Etapa 4 | AutoML e Optuna | `automl` |
| Etapas 5 e 6 | calibração e explicabilidade | `calibracao` |
| Etapa 7 | régua, backtest e estabilidade | `regua` |
| Etapa 8 | PD lifetime (só PD com datas de default) | `lifetime` |

**Pontos de controle.** Cada etapa, ao terminar, guarda o seu resultado no MLflow, como no `99_orquestrador`. Rodar de novo uma célula já concluída **recupera** o resultado em segundos, em vez de recalcular.

**Se uma etapa falhar:** corrija o problema e rode de novo **só aquela célula**. A execução do MLflow continua aberta e as etapas anteriores continuam na memória. Se a sessão cair (as variáveis somem), preencha `RETOMAR_ID_EXECUCAO` com o `id_execucao` e rode o notebook do início: as etapas concluídas são recuperadas dos pontos de controle.

**Ordem:** as células de etapa dependem das anteriores (a etapa 4 usa `selecao`, a 5 usa `automl`...). Rode na ordem.

## 1. Carrega as funções de todas as etapas

Cada `%run` precisa ficar sozinho na célula. O `04_calibracao_explicabilidade` já executa o `03_automl`.

In [ ]:
%run ./00_config

In [ ]:
%run ./01_entrada

In [ ]:
%run ./02_selecao

In [ ]:
%run ./04_calibracao_explicabilidade

In [ ]:
%run ./05_regua

In [ ]:
%run ./06_lifetime

## 2. Parâmetros

Os mesmos do `99_orquestrador`. A base de entrada pode vir de uma tabela (`TABELA_ENTRADA`) ou de um parquet (`CAMINHO_PARQUET`); se as duas estiverem preenchidas, vale o parquet. Filtros e tratamentos da sua base (como deixar `periodo` em minúsculas) ficam na célula da seção 4.

In [ ]:
# -----------------------------------------------------------------------------
# PARÂMETROS DA EXECUÇÃO: edite aqui
# -----------------------------------------------------------------------------
TABELA_ENTRADA = None                                   # ex.: "catalogo.esquema.base_pop_feat"
CAMINHO_PARQUET = None                                  # ex.: "/Volumes/.../2026_pd_imob_features_all"
TAREFA = "pd"                                           # "pd" ou "lgd"
METRICA = "ks"                                          # métrica que escolhe a técnica vencedora no OOT
DATA_CORTE = "2026-09-01"                               # último mês com informação na base, com dia 01
ESQUEMA_SAIDA = "catalogo.esquema"                      # esquema com permissão de escrita
EXPERIMENTO_MLFLOW = "/Workspace/Users/<seu_email>/<pasta>/desafiante_pd"   # nome novo, não uma pasta existente
GRAVAR = True                                           # False = não grava as tabelas de resultado

# Técnicas que o AutoML compara. No serverless, sem "lightgbm".
TECNICAS = ["logistica", "random_forest", "gbt"]

AJUSTES = {
    # "colunas": {"id": "CPF", "safra": "data_ref", "alvo": "target", "amostra": "periodo",
    #             "data_primeiro_default": None, "data_fim_observacao": None},
    # "rotulos_amostra": {"des": "dev", "oot": "oot"},
    # "variaveis": selected_feat_compl,
    # "automl": {"n_tentativas_optuna": 10, "tempo_maximo_optuna_min": 15},
}

# Retomada depois de uma queda de sessão: o id_execucao da execução (None = execução nova).
RETOMAR_ID_EXECUCAO = None
AJUSTES_RETOMADA = {}

## 3. Configuração e execução no MLflow

A primeira célula monta e valida a configuração. A segunda abre a execução no MLflow. Rodar de novo a segunda célula **reabre a mesma execução** (não cria outra), então ela pode ser repetida sem perder os pontos de controle.

In [ ]:
ajustes_execucao = dict(AJUSTES)
if TECNICAS is not None:
    ajustes_execucao["tecnicas"] = TECNICAS
ajustes_execucao.setdefault("esquema_temporario", ESQUEMA_SAIDA)

if RETOMAR_ID_EXECUCAO:
    cfg = carrega_configuracao_da_execucao(EXPERIMENTO_MLFLOW, RETOMAR_ID_EXECUCAO, AJUSTES_RETOMADA)
else:
    cfg = cria_configuracao(tarefa=TAREFA, metrica=METRICA, data_corte=DATA_CORTE,
                            esquema_saida=ESQUEMA_SAIDA if GRAVAR else None,
                            experimento_mlflow=EXPERIMENTO_MLFLOW, ajustes=ajustes_execucao)
print(f"id_execucao = {cfg['id_execucao']}   (use este valor em RETOMAR_ID_EXECUCAO se a sessão cair)")
mostra_configuracao(cfg)

In [ ]:
import time

import pandas as pd

RESUMO_ETAPAS = {}       # uma linha por etapa: origem, tempo e resultado principal


def abre_execucao_mlflow(cfg):
    """
    Abre (ou reabre) a execução do MLflow desta rodada. Na primeira vez, cria a
    execução e guarda o run_id em cfg; nas seguintes, reabre a mesma.
    """
    import mlflow
    if not cfg.get("experimento_mlflow"):
        print("[passo a passo] sem EXPERIMENTO_MLFLOW: nada é registrado e não há pontos de controle.")
        return None
    mlflow.set_experiment(cfg["experimento_mlflow"])
    ativa = mlflow.active_run()
    if ativa is not None:
        if ativa.info.run_id == cfg.get("_run_id_mlflow"):
            print(f"[passo a passo] execução MLflow já aberta: {ativa.info.run_id}")
            return ativa
        mlflow.end_run()
    if cfg.get("_run_id_mlflow"):
        execucao = mlflow.start_run(run_id=cfg["_run_id_mlflow"])
        print(f"[passo a passo] execução MLflow reaberta: {execucao.info.run_id}")
    else:
        execucao = mlflow.start_run(run_name=f"{cfg['tarefa']}_{cfg['id_execucao']}")
        mlflow.set_tags({"id_execucao": cfg["id_execucao"], "tarefa": cfg["tarefa"]})
        registra_configuracao_mlflow(cfg)
        cfg["_run_id_mlflow"] = execucao.info.run_id
        print(f"[passo a passo] execução MLflow criada: {execucao.info.run_id}")
    return execucao


def roda_etapa(chave, nome, executa, recupera, resumo=lambda r: ""):
    """
    Roda uma etapa ou, se ela já tem ponto de controle nesta execução, recupera o resultado.
    Ao terminar uma etapa executada, guarda o ponto de controle.
    """
    inicio = time.time()
    usa_pontos = pontos_de_controle_ativos(cfg)
    if usa_pontos and etapa_concluida(chave):
        print(f"[passo a passo] {nome}: recuperando do ponto de controle...")
        resultado, origem = recupera(le_pacote_ponto(chave)), "recuperada"
    else:
        print(f"[passo a passo] {nome}: iniciando")
        resultado, origem = executa(), "executada"
        if usa_pontos:
            salva_pacote_ponto(chave, resultado)
    minutos = (time.time() - inicio) / 60
    texto = resumo(resultado)
    RESUMO_ETAPAS[chave] = {"etapa": nome, "origem": origem, "tempo_min": round(minutos, 1),
                            "resultado_principal": texto}
    print(f"[passo a passo] {nome} {origem} em {minutos:.1f} min. {texto}")
    return resultado


def mostra(figura):
    """Exibe um gráfico, ou avisa que ele está no MLflow (etapa recuperada de ponto de controle)."""
    if figura is None:
        print("(gráfico de etapa recuperada: veja os artefatos da execução no MLflow)")
    else:
        display(figura)


execucao_mlflow = abre_execucao_mlflow(cfg)

## 4. Base de entrada

Lê o parquet ou a tabela dos parâmetros. Acrescente aqui os tratamentos da sua base (filtros, `drop` de colunas, minúsculas em `periodo`). As datas são levadas ao dia 01 do mês (2026-09-30 vira 2026-09-01). Nada é calculado nesta célula: o Spark só lê quando uma etapa precisa.

In [ ]:
from pyspark.sql import functions as F

if CAMINHO_PARQUET:
    sdf_entrada = spark.read.parquet(CAMINHO_PARQUET)
elif TABELA_ENTRADA:
    sdf_entrada = spark.table(TABELA_ENTRADA)
else:
    raise ValueError("Preencha CAMINHO_PARQUET ou TABELA_ENTRADA nos parâmetros.")

# Tratamentos da sua base (exemplos; ajuste os nomes):
# sdf_entrada = sdf_entrada.filter(F.col("periodo").isin("DEV", "OOT")).drop("cd_cnpj_ger")
# sdf_entrada = sdf_entrada.withColumn("periodo", F.lower(F.trim(F.col("periodo"))))

colunas_de_data = [cfg["colunas"]["safra"]]
if tem_datas_de_evento(cfg):
    colunas_de_data += [cfg["colunas"]["data_primeiro_default"], cfg["colunas"]["data_fim_observacao"]]
sdf_entrada = sdf_entrada.withColumns({c: F.trunc(F.col(c), "month") for c in colunas_de_data})

## 5. Etapa 1: entrada, validação e divisão treino/teste

In [ ]:
entrada = roda_etapa(
    "entrada", "Etapa 1 (entrada)",
    lambda: executa_etapa_entrada(spark, sdf_entrada, cfg, gravar=GRAVAR),
    lambda pacote: recupera_etapa_entrada(spark, cfg, pacote, sdf_entrada),
    lambda r: f"{len(r['variaveis'])} variáveis candidatas.")

exibe(entrada["relatorio_validacao"], "Verificações do contrato de dados")
exibe(entrada["resumo_particao"], "Resumo por partição")

## 6. Etapas 2 e 3: pré-filtros e Boruta

In [ ]:
selecao = roda_etapa(
    "selecao", "Etapas 2–3 (seleção)",
    lambda: executa_etapa_selecao(spark, entrada["base"], entrada["variaveis"], cfg, gravar=GRAVAR),
    lambda pacote: recupera_etapa_selecao(spark, entrada, cfg, pacote),
    lambda r: f"{len(r['variaveis_selecionadas'])} variáveis selecionadas.")

exibe(selecao["funil"], "Funil da seleção de variáveis")
print("Variáveis selecionadas:", selecao["variaveis_selecionadas"])
mostra(selecao.get("figura_boruta"))

## 7. Etapa 4: AutoML e Optuna

Uma linha impressa por técnica e por tentativa do Optuna. Se a célula cair no meio, rodar de novo continua de onde parou: as técnicas avaliadas e as tentativas feitas são recuperadas.

In [ ]:
automl = roda_etapa(
    "automl", "Etapa 4 (AutoML)",
    lambda: executa_etapa_automl(spark, selecao, cfg, gravar=GRAVAR),
    lambda pacote: recupera_etapa_automl(spark, selecao, cfg, pacote),
    lambda r: (f"Final: {r['tecnica_final']} ({r['versao_final']}), {cfg['metrica']} OOT = "
               f"{r['avaliacao_final']['oot'][cfg['metrica']]:.4f}."))

exibe(automl["tabela_comparativa"], "AutoML: todas as técnicas, teste e OOT")
mostra(automl.get("figura_comparacao"))

## 8. Etapas 5 e 6: calibração e explicabilidade

In [ ]:
calibracao = roda_etapa(
    "calibracao", "Etapas 5–6 (calibração e explicabilidade)",
    lambda: executa_etapa_calibracao_explicabilidade(spark, automl, selecao, cfg, gravar=GRAVAR),
    lambda pacote: recupera_etapa_calibracao(spark, automl, cfg, pacote),
    lambda r: (f"Calibração {r['calibracao']['metodo']}; espelho "
               f"{'aprovado' if r['fidelidade']['aprovado'].all() else 'REPROVADO'}."))

exibe(calibracao["tabela_calibracao"], "Calibração: antes × depois")
mostra(calibracao.get("figura_calibracao"))
exibe(calibracao["fidelidade"], "Fidelidade do espelho")
for parte, figura in calibracao.get("figuras_beeswarm", {}).items():
    mostra(figura)
    mostra(calibracao["figuras_importancia"].get(parte))

## 9. Etapa 7: régua, backtest e estabilidade

In [ ]:
regua = roda_etapa(
    "regua", "Etapa 7 (régua e backtest)",
    lambda: executa_etapa_regua(spark, calibracao["base_calibrada"], cfg, gravar=GRAVAR),
    lambda pacote: recupera_etapa_regua(spark, calibracao["base_calibrada"], cfg, pacote),
    lambda r: f"{r['n_faixas_iniciais']} faixas iniciais → {len(r['regua'])} ratings.")

exibe(regua["regua"], "Régua")
mostra(regua.get("figura_estabilidade"))
mostra(regua.get("figura_ratings"))
exibe(regua["backtest"]["rating_periodo"], "Backtest: rating por período")
mostra(regua.get("figura_ponte"))
mostra(regua.get("figura_mapa_status"))

## 10. Etapa 8: PD lifetime

Só roda na PD com as colunas de data do default e de fim de observação.

In [ ]:
lifetime = None
if tem_datas_de_evento(cfg):
    lifetime = roda_etapa(
        "lifetime", "Etapa 8 (PD lifetime)",
        lambda: executa_etapa_lifetime(spark, regua["base_com_rating"], regua["regua"], cfg, gravar=GRAVAR),
        lambda pacote: dict(pacote, contratos_vivos=None, figuras={}),
        lambda r: f"{len(r['curvas'])} curvas.")
    exibe(lifetime["resumo_curvas"], "PD acumulada por rating e horizonte")
    exibe(lifetime["backtest"], "Backtest da curva")
    for nome in ("mapa_safra_idade", "curvas_todos"):
        mostra(lifetime["figuras"].get(nome))
else:
    print("Etapa 8 não se aplica: tarefa LGD, ou base de PD sem as colunas de data de default.")

## 11. Encerramento

Mostra o resumo das etapas, grava-o e fecha a execução do MLflow como concluída.

In [ ]:
resumo_execucao = pd.DataFrame(RESUMO_ETAPAS.values())
exibe(resumo_execucao, "Resumo da execução")
if GRAVAR:
    grava_tabela_delta(spark, resumo_execucao, "resumo_execucao", cfg)
if execucao_mlflow is not None:
    import mlflow
    mlflow.set_tags({"tecnica_final": automl["tecnica_final"], "versao_final": automl["versao_final"]})
    mlflow.log_table(resumo_execucao, "resumo_execucao.json")
    mlflow.end_run(status="FINISHED")
    print(f"[passo a passo] execução MLflow encerrada (id_execucao = {cfg['id_execucao']}).")

## 12. Limpeza

Rode quando terminar de consultar os resultados: apaga as tabelas temporárias da execução. Depois dela, os DataFrames Spark das variáveis acima deixam de funcionar; as tabelas pandas e os gráficos continuam.

In [ ]:
apaga_tabelas_temporarias(spark, cfg, incluir_pontos_de_controle=True)